# Phase 2 — the model writes trading strategies (code-as-action)

One episode: the model reads a fixed task spec (it never sees market data) and writes
an `allocate(prices) -> weights` function. We execute it in a sandbox, walking it
forward over a **hidden random 3-year window** of 2008-2019, and the reward is the
window's **annualized Sharpe net of costs**. GRPO groups share a window; windows vary
across groups, so era-fragile logic loses on average.

**Calibration (30 random windows, from `README` / local runs):** equal-weight scores
mean **+0.81**, 60/40 **+1.29** (beats EW in 90% of windows), hardcoded train-era
tangency **+1.40** (97%). So: real headroom above 1/N, and the grader's argmax is
era-memorization — hence the pre-registered branches:
1. (base case) converges to bond-tilted ~static programs -> val re-runs the 1b story
2. adaptive risk-managed logic that transfers to val better than the memorizers
3. equal-weight/degenerate collapse (unlikely: the 90%-win gradient points away)

Model: **Qwen2.5-Coder-14B-Instruct** (code quality matters now). Runtime: A100,
ideally 80GB (on 40GB: set `gpu_memory_utilization=0.55`). ~2h, ~15 units.

In [ ]:
%%capture
!pip install unsloth vllm
!pip install yfinance pyarrow

In [ ]:
# --- 1. Code + data + the task spec the model will see -------------------------
import os, sys
if not os.path.exists("rl-finance"):
    !git clone https://github.com/marcnasrisme/rl-finance.git
sys.path.insert(0, "rl-finance/src")

from pathlib import Path
import numpy as np
import pandas as pd

from rl_finance.data import SPLITS, UNIVERSE, download_prices
from rl_finance.codegen import (
    PROMPT, make_code_reward, make_dataset_rows, make_windows,
    run_strategy, score_strategy, weekly_metrics,
)
from rl_finance.sandbox import StrategyError, clean_weights, compile_strategy

prices = download_prices(cache=Path("rl-finance/data/prices.parquet"))
print(PROMPT)  # this exact text is every episode's prompt — no market data in it

In [ ]:
# --- 2. Model: a 14B coder + LoRA ----------------------------------------------
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-14B-Instruct",
    max_seq_length=2048,          # ~700-token prompt + <=768-token completion
    load_in_4bit=False,           # bf16 on 80GB (set True + util 0.55 on 40GB)
    fast_inference=True,
    max_lora_rank=32,
    gpu_memory_utilization=0.7,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# --- 3. ZERO-SHOT baseline: what does an untrained 14B coder write? ------------
# 8 samples at temp 0.8, each graded on the SAME fixed pool of 10 train windows
# (mean score). This is the bar RL must beat — and it's genuinely uncertain
# whether a good coder zero-shots past equal-weight's +0.81.
from vllm import SamplingParams

EVAL_POOL = make_windows(prices, 10, seed=123)

def grade_on_pool(fn):
    return float(np.mean([score_strategy(fn, prices, s, e) for s, e in EVAL_POOL]))

def sample_and_grade(n, temperature, lora_request=None, label=""):
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT}], tokenize=False, add_generation_prompt=True)
    outs = model.fast_generate(
        [chat] * n,
        sampling_params=SamplingParams(temperature=temperature, max_tokens=768),
        lora_request=lora_request,
    )
    graded = []
    for i, o in enumerate(outs):
        text = o.outputs[0].text
        try:
            score = grade_on_pool(compile_strategy(text))
        except StrategyError as e:
            score, text = None, text + f"\n# INVALID: {e}"
        graded.append((score, text))
        print(f"[{label}{i}] {'INVALID' if score is None else f'{score:+.2f}'}")
    return graded

zero_shot = sample_and_grade(8, temperature=0.8, label="zs")
valid = [g for g in zero_shot if g[0] is not None]
best_zs_score, best_zs_code = max(valid, key=lambda g: g[0])
print(f"\nbest zero-shot: {best_zs_score:+.2f} (equal-weight bar: +0.81)\n")
print(best_zs_code)

In [ ]:
# --- 4. Episodes + Drive ---------------------------------------------------------
# WINDOWS_PER_EPISODE=1: reward = the window's Sharpe.
# WINDOWS_PER_EPISODE=2: minimax — reward = WORST of two windows from different
# eras; punishes era-memorization harder. Run 1 with 1; A/B with 2 if branch 1 hits.
from datasets import Dataset
from google.colab import drive

WINDOWS_PER_EPISODE = 1
N_EPISODES = 400

rows = make_dataset_rows(prices, N_EPISODES, seed=7,
                         windows_per_episode=WINDOWS_PER_EPISODE)
train_ds = Dataset.from_list(rows)

drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/rl-finance/outputs_p2"
!mkdir -p {CKPT_DIR}
train_ds

In [ ]:
# --- 5. GRPO training -------------------------------------------------------------
# temp 0.9 (not 1.0): code needs coherence; diversity still ample.
# 400 episodes x 8 generations / 16 per step = 200 optimizer steps, 1 epoch.
# Reward: sandboxed walk-forward Sharpe (clip [-3, 6]); crash/timeout/contract
# violation = -5. Watch: mean reward vs the +0.81 EW bar and +1.29 60/40 bar;
# invalid share (expect high-ish early, falling fast); reward_std > 0.
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=CKPT_DIR,
    save_steps=25,
    save_total_limit=3,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_generations=8,
    max_prompt_length=1024,
    max_completion_length=768,
    temperature=0.9,
    num_train_epochs=1,
    logging_steps=5,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[make_code_reward(prices)],
    args=config,
    train_dataset=train_ds,
)
trainer.train()

In [ ]:
# --- 6. VERDICT on val (2020-2022) -------------------------------------------------
# Sample 4 strategies from the trained model, select the best ON TRAIN windows
# (never select on val!), then run that single program through the same daily
# backtester as every benchmark. Compare: trained vs zero-shot-best vs classics.
model.save_lora("grpo_coder_lora")
!cp -r grpo_coder_lora /content/drive/MyDrive/rl-finance/

trained = sample_and_grade(4, temperature=0.3,
                           lora_request=model.load_lora("grpo_coder_lora"),
                           label="tr")
valid_tr = [g for g in trained if g[0] is not None]
best_tr_score, best_tr_code = max(valid_tr, key=lambda g: g[0])
print(f"\nselected trained program (train-pool {best_tr_score:+.2f}):\n")
print(best_tr_code)

# wrap a generated function as a backtester policy (explicit CASH remainder!)
from rl_finance.backtest import run_backtest
from rl_finance.benchmarks import BENCHMARKS
from rl_finance.metrics import format_table
from rl_finance.data import CASH

P = prices.reset_index(drop=True)
DATE_POS = {d: i for i, d in enumerate(prices.index)}

def as_policy(code_text):
    fn = compile_strategy(code_text)
    def policy(date, feats):
        try:
            w = clean_weights(fn(P.iloc[: DATE_POS[date] + 1]), UNIVERSE)
        except Exception:
            w = {}
        w[CASH] = 1.0 - sum(w.values())  # run_backtest needs weights summing to 1
        return w
    return policy

start, end = SPLITS["val"]
results = {
    "code_grpo": run_backtest(as_policy(best_tr_code), prices, start, end)["metrics"],
    "code_zero_shot": run_backtest(as_policy(best_zs_code), prices, start, end)["metrics"],
}
for name, policy in BENCHMARKS.items():
    results[name] = run_backtest(policy, prices, start, end)["metrics"]
print(format_table(results))